# Creditcard Fraud Detection

**Project Overview:** Using a dataset of 284,807 credit card transactions where only 0.17% are fraudulent, the goal is to build a model that reliably catches fraud without flagging too many legitimate transactions.


In [1]:
import pandas as pd
df = pd.read_csv("data/creditcard.csv")
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True).round(4))
print(df['Class'].value_counts()/len(df)) 

Class
0    284315
1       492
Name: count, dtype: int64
Class
0    0.9983
1    0.0017
Name: proportion, dtype: float64
Class
0    0.998273
1    0.001727
Name: count, dtype: float64


In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum().sum())
df.head()

(284807, 31)
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
0


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


There is a severe class imbalance (578:1). The columns are PCA-transformed for privacy. Class 1 represents frauds while 0 is legimate transactions.

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train class distribution:')
print(y_train.value_counts())

Train class distribution:
Class
0    227451
1       394
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

There is a severe imbalance between classes and needs to be addressed by using SMOTE. I will use sampling strategy 0.1 to keep it realistic.

In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:')
print(y_train.value_counts())
print('\nAfter SMOTE:')
print(pd.Series(y_train_resampled).value_counts())

Before SMOTE:
Class
0    227451
1       394
Name: count, dtype: int64

After SMOTE:
Class
0    227451
1     22745
Name: count, dtype: int64


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_resampled, y_train_resampled)

y_pred = model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[56853    11]
 [   16    82]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.84      0.86        98

    accuracy                           1.00     56962
   macro avg       0.94      0.92      0.93     56962
weighted avg       1.00      1.00      1.00     56962



- Caught 82 out of 98 real frauds (recall=84%)
- Missed 16 real frauds (false negatives)
- wrongly flagged 11 legitimate transactions

Missing fraud transactions are much more deadly so maximizing the recall is the priority.

In [11]:
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

y_pred_adjusted = (y_pred_proba >= 0.3).astype(int)

print(classification_report(y_test, y_pred_adjusted))
print(confusion_matrix(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.71      0.89      0.79        98

    accuracy                           1.00     56962
   macro avg       0.85      0.94      0.89     56962
weighted avg       1.00      1.00      1.00     56962

[[56828    36]
 [   11    87]]


Lowering threshold makes the model more agressive at flagging fraud. 
- Missed frauds decreased from 16 to 11
- Wrong flags increased from 11 to 36

For a bank, catching 5 more frauds at the cost of 25 more false alerts is worthy tradeoff.